## CS466: Data Preprocessing
- data overview
- examples of data
- vectorization
-

In [1]:
!pip -q install pandas pyarrow pillow torch torchvision transformers tqdm google-cloud-storage kagglehub tiktoken chromadb openai

In [2]:
!pip install huggingface_hub
from huggingface_hub import notebook_login

notebook_login()


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [3]:
from google.colab import auth
auth.authenticate_user()

from google.cloud import storage
client = storage.Client()
print("Authenticated successfully.")

Authenticated successfully.


In [4]:
import os
import json
import zipfile
import random
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch
from transformers import CLIPProcessor, CLIPModel
import kagglehub
import tiktoken

In [5]:
import random
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive')

SEED = 42
random.seed(SEED)

# ----------------------------
# GCS bucket + object names
# ----------------------------
BUCKET_NAME = "cs466_bucket_1"

# VISPR in GCS
VISPR_IMAGE_TAR = "VISPR/vispr_images.tar.gz"
VISPR_ANN_TAR = "VISPR/vispr_anno.tar.gz"

# BIV-Priv in GCS
BIV_IMAGE_ZIP_BLOB = "Biv_Priv/query_images.zip"
BIV_ANN_JSON_BLOB = "Biv_Priv/query_set_images_info.json"

# PANORAMA
PANORAMA_URI = "hf://datasets/srirxml/PANORAMA/data/train-00000-of-00001.parquet"

# ----------------------------
# Local working dirs in Colab runtime
# ----------------------------
BASE_DIR = Path("/content/cs466_baseline_demo")
BASE_DIR.mkdir(parents=True, exist_ok=True)

RAW_DIR = BASE_DIR / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

EXTRACT_DIR = BASE_DIR / "extracted"
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

# ----------------------------
# Persistent dirs in Google Drive
# ----------------------------
DRIVE_BASE_DIR = Path("/content/drive/MyDrive/cs466_baseline_demo")
DRIVE_BASE_DIR.mkdir(parents=True, exist_ok=True)

OUT_DIR = DRIVE_BASE_DIR / "outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

EMB_DIR = OUT_DIR / "image_embeddings"
EMB_DIR.mkdir(parents=True, exist_ok=True)


# Sample sizes for quick baseline
N_PANORAMA = 2000
N_DATASIR = 2000
N_VISPR = 2000
N_BIVPRIV = 2112  # uses the full 2,112 — just cap at len(members)

print("Local BASE_DIR:", BASE_DIR)
print("Persistent DRIVE_BASE_DIR:", DRIVE_BASE_DIR)
print("Persistent OUT_DIR:", OUT_DIR)
print("Persistent EMB_DIR:", EMB_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Local BASE_DIR: /content/cs466_baseline_demo
Persistent DRIVE_BASE_DIR: /content/drive/MyDrive/cs466_baseline_demo
Persistent OUT_DIR: /content/drive/MyDrive/cs466_baseline_demo/outputs
Persistent EMB_DIR: /content/drive/MyDrive/cs466_baseline_demo/outputs/image_embeddings


In [6]:
def stable_id(prefix: str, text: str) -> str:
    digest = hashlib.md5(text.encode("utf-8")).hexdigest()[:12]
    return f"{prefix}_{digest}"

def read_json(path: Path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def save_jsonl(records, path: Path):
    with open(path, "w", encoding="utf-8") as f:
        for record in records:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")

def print_sample(title: str, obj, max_chars: int = 1200):
    print("=" * 80)
    print(title)
    print("-" * 80)
    if isinstance(obj, str):
        s = obj
    else:
        s = json.dumps(obj, ensure_ascii=False, indent=2)
    print(s[:max_chars] + ("..." if len(s) > max_chars else ""))
    print()

def download_blob(bucket_name: str, source_blob_name: str, destination_file_name: str):
    bucket = client.bucket(bucket_name)
    blob = bucket.blob(source_blob_name)
    blob.download_to_filename(destination_file_name)
    print(f"Downloaded gs://{bucket_name}/{source_blob_name} -> {destination_file_name}")

def list_zip_members(zip_path: Path, suffixes=(".jpg", ".jpeg", ".png", ".json")):
    with zipfile.ZipFile(zip_path, "r") as zf:
        members = [m for m in zf.namelist() if m.lower().endswith(suffixes)]
    return members

def extract_sample_from_zip(zip_path: Path, members, out_dir: Path, n: int, seed: int = 42):
    out_dir.mkdir(parents=True, exist_ok=True)
    random.seed(seed)
    sampled = random.sample(members, min(n, len(members)))
    extracted_paths = []

    with zipfile.ZipFile(zip_path, "r") as zf:
        for member in sampled:
            target_path = out_dir / Path(member).name
            with zf.open(member) as src, open(target_path, "wb") as dst:
                dst.write(src.read())
            extracted_paths.append(target_path)

    return extracted_paths

def chunk_text(text: str, chunk_tokens: int = 300, overlap_tokens: int = 50) -> list[str]:
    """Token-aware chunker using tiktoken (cl100k_base, same vocab as text-embedding-3-small)."""
    enc = tiktoken.get_encoding("cl100k_base")
    token_ids = enc.encode(text)
    chunks = []
    start = 0
    while start < len(token_ids):
        end = min(len(token_ids), start + chunk_tokens)
        chunks.append(enc.decode(token_ids[start:end]))
        if end == len(token_ids):
            break
        start = end - overlap_tokens
    return chunks

In [7]:
device = "cuda" if torch.cuda.is_available() else "cpu"

clip_model_name = "openai/clip-vit-base-patch32"
clip_processor = CLIPProcessor.from_pretrained(clip_model_name)
clip_model = CLIPModel.from_pretrained(clip_model_name).to(device)
clip_model.eval()

print("Using device:", device)

The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Using device: cuda


In [8]:
@torch.no_grad()
def encode_image_with_clip(image_path: str):
    image = Image.open(image_path).convert("RGB")
    inputs = clip_processor(images=image, return_tensors="pt")
    pixel_values = inputs["pixel_values"].to(device)
    image_features = clip_model.get_image_features(pixel_values=pixel_values)
    image_features = image_features / image_features.norm(dim=-1, keepdim=True)
    return image_features[0].cpu().numpy()

In [9]:
from pathlib import Path

TMP_DIR = Path("/content/demo_data")
TMP_DIR.mkdir(exist_ok=True)

def download_blob(bucket_name, source_blob_name, destination_file_name):
    bucket = client.bucket(bucket_name)
    blob = bucket.blob(source_blob_name)
    blob.download_to_filename(destination_file_name)
    print(f"Downloaded gs://{bucket_name}/{source_blob_name} -> {destination_file_name}")

In [10]:
BIV_IMAGE_ZIP_PATH = TMP_DIR / "bivpriv_query_images.zip"
BIV_ANNO_ZIP_PATH = TMP_DIR / "bivpriv_query_info.json"

if not BIV_ANNO_ZIP_PATH.exists():
  download_blob(
      "cs466_bucket_1",
      "Biv_Priv/query_set_images_info.json",
      str(BIV_ANNO_ZIP_PATH)
  )

if not BIV_IMAGE_ZIP_PATH.exists():
  download_blob(
      "cs466_bucket_1",
      "Biv_Priv/query_images.zip",
      str(BIV_IMAGE_ZIP_PATH)
  )

Downloaded gs://cs466_bucket_1/Biv_Priv/query_set_images_info.json -> /content/demo_data/bivpriv_query_info.json
Downloaded gs://cs466_bucket_1/Biv_Priv/query_images.zip -> /content/demo_data/bivpriv_query_images.zip


In [11]:
VISPR_IMAGE_TAR_PATH = RAW_DIR / "vispr_images.tar.gz"
VISPR_ANNO_TAR_PATH = RAW_DIR / "vispr_anno.tar.gz"

if not VISPR_ANNO_TAR_PATH.exists():
    download_blob(
        BUCKET_NAME,
        "VISPR/vispr_anno.tar.gz",
        str(VISPR_ANNO_TAR_PATH)
    )

if not VISPR_IMAGE_TAR_PATH.exists():
    download_blob(
        BUCKET_NAME,
        "VISPR/vispr_images.tar.gz",
        str(VISPR_IMAGE_TAR_PATH)
    )

Downloaded gs://cs466_bucket_1/VISPR/vispr_anno.tar.gz -> /content/cs466_baseline_demo/raw/vispr_anno.tar.gz
Downloaded gs://cs466_bucket_1/VISPR/vispr_images.tar.gz -> /content/cs466_baseline_demo/raw/vispr_images.tar.gz


In [12]:
import json

with open(TMP_DIR / "bivpriv_query_info.json", "r", encoding="utf-8") as f:
    biv_info = json.load(f)

print(type(biv_info))
if isinstance(biv_info, dict):
    print("Top-level keys:", list(biv_info.keys()))
elif isinstance(biv_info, list):
    print("Number of entries:", len(biv_info))

print(json.dumps(biv_info, indent=2)[:10])

<class 'dict'>
Top-level keys: ['categories', 'annotations', 'images']
{
  "categ


In [13]:
EXTRACT_DIR_BIV = TMP_DIR / "bivpriv_extracted"
EXTRACT_DIR_BIV.mkdir(exist_ok=True)

with zipfile.ZipFile(TMP_DIR / "bivpriv_query_images.zip", "r") as zf:
    members = [
    m for m in zf.namelist()
    if m.lower().endswith((".jpg", ".jpeg", ".png"))
    and not m.startswith("__MACOSX")
    and not Path(m).name.startswith("._")
]

print("Total image files in zip:", len(members))
print("First few members:", members[:5])

Total image files in zip: 1056
First few members: ['query_images/723.jpeg', 'query_images/689.jpeg', 'query_images/373.jpeg', 'query_images/666.jpeg', 'query_images/236.jpeg']


In [14]:
bivpriv_sampled_images = []

random.seed(SEED)
sample_members = random.sample(members, min(N_BIVPRIV, len(members)))

with zipfile.ZipFile(TMP_DIR / "bivpriv_query_images.zip", "r") as zf:
    for member in sample_members:
        out_path = EXTRACT_DIR_BIV / Path(member).name
        with zf.open(member) as src, open(out_path, "wb") as dst:
            dst.write(src.read())
        bivpriv_sampled_images.append(out_path)

print("Extracted BIV-Priv sample:", len(bivpriv_sampled_images))

Extracted BIV-Priv sample: 1056


In [15]:
import tarfile

def list_tar_members(tar_path, suffixes=(".jpg", ".jpeg", ".png", ".json")):
    members = []
    with tarfile.open(tar_path, "r:gz") as tar:
        for m in tar.getmembers():
            if m.isfile() and m.name.lower().endswith(suffixes):
                members.append(m.name)
    return members

vispr_image_members = list_tar_members(RAW_DIR / "vispr_images.tar.gz", suffixes=(".jpg", ".png"))
vispr_ann_members = list_tar_members(RAW_DIR / "vispr_anno.tar.gz", suffixes=(".json",))

print("VISPR image count:", len(vispr_image_members))
print("VISPR annotation count:", len(vispr_ann_members))
print("Sample image paths:", vispr_image_members[:5])

VISPR image count: 4167
VISPR annotation count: 4167
Sample image paths: ['val2017/2017_14605178.jpg', 'val2017/2017_87026020.jpg', 'val2017/2017_24230923.jpg', 'val2017/2017_47117897.jpg', 'val2017/2017_90552025.jpg']


In [16]:
bivpriv_ann = read_json(TMP_DIR / "bivpriv_query_info.json")

print("BIV-Priv annotation type:", type(bivpriv_ann))
if isinstance(bivpriv_ann, dict):
    print("Top-level keys:", list(bivpriv_ann.keys()))
elif isinstance(bivpriv_ann, list):
    print("Number of entries:", len(bivpriv_ann))

print_sample("BIV-Priv annotation sample", bivpriv_ann)

BIV-Priv annotation type: <class 'dict'>
Top-level keys: ['categories', 'annotations', 'images']
BIV-Priv annotation sample
--------------------------------------------------------------------------------
{
  "categories": [
    {
      "isthing": 1,
      "is_crowd": 0,
      "id": 116,
      "name": "local_newspaper",
      "fold": 4
    },
    {
      "isthing": 1,
      "is_crowd": 0,
      "id": 115,
      "name": "bank_statement",
      "fold": 4
    },
    {
      "isthing": 1,
      "is_crowd": 0,
      "id": 114,
      "name": "bills_or_receipt",
      "fold": 4
    },
    {
      "isthing": 1,
      "is_crowd": 0,
      "id": 113,
      "name": "business_card",
      "fold": 4
    },
    {
      "isthing": 1,
      "is_crowd": 0,
      "id": 112,
      "name": "condom_box",
      "fold": 4
    },
    {
      "isthing": 1,
      "is_crowd": 0,
      "id": 111,
      "name": "credit_or_debit_card",
      "fold": 4
    },
    {
      "isthing": 1,
      "is_crowd": 0,
      "id"

In [17]:
def load_json_from_tar(tar_path):
    records = []
    with tarfile.open(tar_path, "r:gz") as tar:
        for member in tar.getmembers():
            if member.isfile() and member.name.endswith(".json"):
                f = tar.extractfile(member)
                if f:
                    records.append(json.load(f))
    return records

vispr_annotations = load_json_from_tar(RAW_DIR / "vispr_anno.tar.gz")

print("Loaded VISPR annotations:", len(vispr_annotations))
print_sample("VISPR annotation example", vispr_annotations[0])

Loaded VISPR annotations: 4167
VISPR annotation example
--------------------------------------------------------------------------------
{
  "id": "2017_10735550",
  "image_path": "images/val2017/2017_10735550.jpg",
  "labels": [
    "a0_safe"
  ],
  "openimages_id": "4bb51c80f97a81d6",
  "source_url": "https://farm7.staticflickr.com/4028/4330838526_c52a02b736_o.jpg"
}



In [18]:
import random
import tarfile
from pathlib import Path
from tqdm.auto import tqdm

def extract_sample_from_tar_fast(tar_path, members, out_dir, n=20, seed=42):
    out_dir.mkdir(parents=True, exist_ok=True)
    random.seed(seed)

    sampled = random.sample(members, min(n, len(members)))
    sampled_set = set(sampled)

    extracted_paths = []

    with tarfile.open(tar_path, "r:gz") as tar:
        all_members = tar.getmembers()

        for member in tqdm(all_members, desc="Scanning tar"):
            if member.name not in sampled_set:
                continue

            f = tar.extractfile(member)
            if f is None:
                continue

            out_path = out_dir / Path(member.name).name
            with open(out_path, "wb") as dst:
                dst.write(f.read())

            extracted_paths.append(out_path)

            if len(extracted_paths) == len(sampled):
                break

    return extracted_paths

In [19]:
vispr_sampled_images = extract_sample_from_tar_fast(
    RAW_DIR / "vispr_images.tar.gz",
    vispr_image_members,
    EXTRACT_DIR / "vispr",
    n=N_VISPR,
    seed=SEED
)

print("Extracted VISPR sample:", len(vispr_sampled_images))

Scanning tar:   0%|          | 0/4168 [00:00<?, ?it/s]

Extracted VISPR sample: 2000


In [20]:
panorama_df = pd.read_parquet(PANORAMA_URI)

print("PANORAMA shape:", panorama_df.shape)
print("PANORAMA columns:", list(panorama_df.columns))
display(panorama_df.head(3))

PANORAMA shape: (384794, 3)
PANORAMA columns: ['id', 'content-type', 'text']


,id,content-type,text
0,4f7dc93c-14bf-4f47-bd89-84a36a7e1f62,Article,"Raymond Phillips (born July 1, 1956, in Tanya ..."
1,4f7dc93c-14bf-4f47-bd89-84a36a7e1f62,Social Media,Raymond : Just wrapped up a productive morning...
2,4f7dc93c-14bf-4f47-bd89-84a36a7e1f62,Social Media,Raymond : Enjoying a chill afternoon with Morg...


In [21]:
candidate_text_cols = ["text", "content", "body", "document", "article", "post"]

text_col = None
for c in candidate_text_cols:
    if c in panorama_df.columns:
        text_col = c
        break

if text_col is None:
    for c in panorama_df.columns:
        if panorama_df[c].dtype == "object":
            non_null = panorama_df[c].dropna()
            if len(non_null) > 0 and isinstance(non_null.iloc[0], str):
                text_col = c
                break

print("Chosen PANORAMA text column:", text_col)
assert text_col is not None, "Could not find a PANORAMA text column."

Chosen PANORAMA text column: text


In [22]:
panorama_sample = panorama_df.dropna(subset=[text_col]).sample(
    n=min(N_PANORAMA, len(panorama_df)),
    random_state=SEED
).reset_index(drop=True)

text_records = []

for _, row in panorama_sample.iterrows():
    text = str(row[text_col]).strip()
    record_id = stable_id("panorama", text)

    text_records.append({
        "record_id": record_id,
        "modality": "text",
        "content": text,
        "source_dataset": "PANORAMA",
        "metadata": {}
    })

print("Built PANORAMA text records:", len(text_records))
print_sample("PANORAMA text record sample", text_records[0])

Built PANORAMA text records: 2000
PANORAMA text record sample
--------------------------------------------------------------------------------
{
  "record_id": "panorama_9affe9444ceb",
  "modality": "text",
  "content": "Name: Declan  \nLocation: Lake Stuartview  \nTimestamp: 2023-09-20 11:13  \nReading about innovative design approaches always sparks ideas in my own projects. Growing up in Lake Stuartview and now working in Boothfort gives me a unique perspective on integrating regional influences with modern trends. Excellent post!",
  "source_dataset": "PANORAMA",
  "metadata": {}
}



In [23]:
datasir_path = kagglehub.dataset_download(
    "fanmo1/datasir",
    path="DataSIR.json"
)

datasir_data = read_json(Path(datasir_path))
print(f"Successfully loaded {len(datasir_data)} DataSIR records")
print_sample("DataSIR raw example", datasir_data[0] if isinstance(datasir_data, list) and len(datasir_data) > 0 else datasir_data)

Using Colab cache for faster access to the 'datasir' dataset.
Successfully loaded 1647501 DataSIR records
DataSIR raw example
--------------------------------------------------------------------------------
{
  "Data_type": "General",
  "Sensitive_data": "IMEI",
  "Original_data": "898264829382206",
  "Encoding_method": "Octal",
  "Encoded_data": "10 11 10 2 6 4 10 2 11 3 10 2 2 0 6"
}



In [30]:
def datasir_pick_text(record):
    for key in ["Encoded_data", "Original_data", "transformed_text", "text", "content", "value", "sample"]:
        if key in record and isinstance(record[key], str) and record[key].strip():
            return record[key]
    return None

datasir_records = []

if isinstance(datasir_data, list):
    sampled_datasir = random.sample(datasir_data, min(N_DATASIR, len(datasir_data)))
else:
    sampled_datasir = []

for rec in sampled_datasir:
    text = datasir_pick_text(rec)
    if not text:
        continue

    record_id = stable_id("datasir", text)
    datasir_records.append({
        "record_id": record_id,
        "modality": "text",
        "content": text,
        "source_dataset": "DataSIR",
        "metadata": {
            "is_obfuscated": True
        }
    })

print("Built DataSIR text records:", len(datasir_records))
if datasir_records:
    print_sample("DataSIR text record sample", datasir_records[0])

Built DataSIR text records: 1981
DataSIR text record sample
--------------------------------------------------------------------------------
{
  "record_id": "datasir_cc0645e0c698",
  "modality": "text",
  "content": "⠠⠚⠠⠙⠠⠃⠠⠉⠱⠠⠍⠠⠽⠠⠎⠠⠟⠠⠇⠱⠌⠌⠠⠎⠠⠟⠠⠇⠤⠼⠁⠼⠚⠼⠁⠨⠠⠇⠠⠕⠠⠉⠠⠁⠠⠇⠨⠠⠉⠠⠕⠠⠍⠱⠼⠉⠼⠃⠼⠋⠼⠉⠌⠠⠙⠠⠃⠸⠠⠏⠠⠗⠠⠕⠠⠙⠼⠃⠼⠚⠼⠃⠼⠛⠹⠠⠏⠠⠕⠠⠕⠠⠇⠠⠎⠠⠊⠠⠵⠠⠑⠿⠼⠁⠼⠉⠯⠠⠉⠠⠓⠠⠁⠠⠗⠠⠎⠠⠑⠠⠞⠿⠠⠥⠠⠞⠠⠋⠼⠓⠯⠠⠥⠠⠎⠠⠑⠠⠎⠠⠎⠠⠇⠿⠠⠋⠠⠁⠠⠇⠠⠎⠠⠑",
  "source_dataset": "DataSIR",
  "metadata": {
    "is_obfuscated": true
  }
}



In [25]:
vispr_ann_by_filename = {}

for ann in vispr_annotations:
    image_path = ann.get("image_path", "")
    filename = Path(image_path).name
    vispr_ann_by_filename[filename] = ann

vispr_pairs = []
for img_path in vispr_sampled_images:
    ann = vispr_ann_by_filename.get(img_path.name)
    if ann is not None:
        vispr_pairs.append((img_path, ann))

print("Matched VISPR image-annotation pairs:", len(vispr_pairs))
if vispr_pairs:
    print_sample("VISPR matched annotation sample", vispr_pairs[0][1])

Matched VISPR image-annotation pairs: 2000
VISPR matched annotation sample
--------------------------------------------------------------------------------
{
  "id": "2017_87026020",
  "image_path": "images/val2017/2017_87026020.jpg",
  "labels": [
    "a0_safe"
  ],
  "openimages_id": "c28f77bda9fb9441",
  "source_url": "https://farm2.staticflickr.com/4128/5164135151_5f256a1b3c_o.jpg"
}



In [26]:
biv_image_to_labels = {}

if isinstance(bivpriv_ann, dict) and "annotations" in bivpriv_ann and "categories" in bivpriv_ann:
    biv_categories = {c["id"]: c["name"] for c in bivpriv_ann.get("categories", [])}
    for ann in bivpriv_ann.get("annotations", []):
        img_id = ann.get("image_id")
        cat_id = ann.get("category_id")
        cat_name = biv_categories.get(cat_id, f"cat_{cat_id}")
        biv_image_to_labels.setdefault(img_id, []).append(cat_name)

    biv_image_info = {img["id"]: img for img in bivpriv_ann.get("images", [])}
else:
    biv_image_info = {}

print("Parsed BIV-Priv image metadata entries:", len(biv_image_info))
print("Parsed BIV-Priv image label mappings:", len(biv_image_to_labels))

Parsed BIV-Priv image metadata entries: 1056
Parsed BIV-Priv image label mappings: 0


In [27]:
@torch.no_grad()
def encode_image_with_clip(image_path: str):
    image = Image.open(image_path).convert("RGB")
    inputs = clip_processor(images=image, return_tensors="pt")
    pixel_values = inputs["pixel_values"].to(device)
    output = clip_model.get_image_features(pixel_values=pixel_values)

    # handle both tensor and wrapped output object
    if hasattr(output, "pooler_output"):
        image_features = output.pooler_output
    elif hasattr(output, "last_hidden_state"):
        image_features = output.last_hidden_state[:, 0, :]  # CLS token
    else:
        image_features = output  # already a tensor

    image_features = image_features / image_features.norm(dim=-1, keepdim=True)
    return image_features[0].cpu().numpy()

In [28]:
image_records = []

for image_path, ann in tqdm(vispr_pairs, desc="Processing VISPR"):
    record_id = stable_id("vispr", ann.get("id", image_path.name))

    emb = encode_image_with_clip(str(image_path))
    emb_path = EMB_DIR / f"{record_id}.npy"
    np.save(emb_path, emb)

    image_records.append({
        "record_id": record_id,
        "modality": "image",
        "image_path": str(image_path),
        "clip_embedding_path": str(emb_path),
        "source_dataset": "VISPR",
        "visual_tags": ann.get("labels", []),
        "metadata": {
            "annotation_id": ann.get("id"),
            "openimages_id": ann.get("openimages_id"),
            "source_url": ann.get("source_url")
        }
    })

print("VISPR image records built:", len(image_records))
if image_records:
    print_sample("VISPR image record sample", image_records[0])

Processing VISPR:   0%|          | 0/2000 [00:00<?, ?it/s]

VISPR image records built: 2000
VISPR image record sample
--------------------------------------------------------------------------------
{
  "record_id": "vispr_695fd91a54ee",
  "modality": "image",
  "image_path": "/content/cs466_baseline_demo/extracted/vispr/2017_87026020.jpg",
  "clip_embedding_path": "/content/drive/MyDrive/cs466_baseline_demo/outputs/image_embeddings/vispr_695fd91a54ee.npy",
  "source_dataset": "VISPR",
  "visual_tags": [
    "a0_safe"
  ],
  "metadata": {
    "annotation_id": "2017_87026020",
    "openimages_id": "c28f77bda9fb9441",
    "source_url": "https://farm2.staticflickr.com/4128/5164135151_5f256a1b3c_o.jpg"
  }
}



In [31]:
for image_path in tqdm(bivpriv_sampled_images, desc="Processing BIV-Priv"):
    record_id = stable_id("bivpriv", image_path.name)

    emb = encode_image_with_clip(str(image_path))
    emb_path = EMB_DIR / f"{record_id}.npy"
    np.save(emb_path, emb)

    # simple fallback: no exact label mapping if filename/image_id link is unavailable
    labels = []

    image_records.append({
        "record_id": record_id,
        "modality": "image",
        "image_path": str(image_path),
        "clip_embedding_path": str(emb_path),
        "source_dataset": "BIV-Priv",
        "visual_tags": labels,
        "metadata": {}
    })

print("Total image records after BIV-Priv:", len(image_records))
if image_records:
    print_sample("BIV-Priv image record sample", image_records[-1])

Processing BIV-Priv:   0%|          | 0/1056 [00:00<?, ?it/s]

Total image records after BIV-Priv: 3071
BIV-Priv image record sample
--------------------------------------------------------------------------------
{
  "record_id": "bivpriv_3016a2ffb5be",
  "modality": "image",
  "image_path": "/content/demo_data/bivpriv_extracted/47.jpeg",
  "clip_embedding_path": "/content/drive/MyDrive/cs466_baseline_demo/outputs/image_embeddings/bivpriv_3016a2ffb5be.npy",
  "source_dataset": "BIV-Priv",
  "visual_tags": [],
  "metadata": {}
}



In [32]:
all_text_records = text_records + datasir_records

save_jsonl(all_text_records, OUT_DIR / "text_records.jsonl")
save_jsonl(image_records, OUT_DIR / "image_records.jsonl")

print("Saved text_records.jsonl and image_records.jsonl")

Saved text_records.jsonl and image_records.jsonl


In [33]:
retrieval_units = []

# text retrieval units
for rec in all_text_records:
    chunks = chunk_text(rec["content"])
    for i, chunk in enumerate(chunks):
        retrieval_units.append({
            "unit_id": f"{rec['record_id']}_chunk_{i}",
            "record_id": rec["record_id"],
            "modality": "text",
            "retrieval_text": chunk,
            "source_dataset": rec["source_dataset"],
            "metadata": rec.get("metadata", {})
        })

# image retrieval units
for rec in image_records:
    tag_text = ", ".join(rec.get("visual_tags", [])) if rec.get("visual_tags") else "image"
    retrieval_units.append({
        "unit_id": f"{rec['record_id']}_img",
        "record_id": rec["record_id"],
        "modality": "image",
        "retrieval_text": f"{rec['source_dataset']} image with tags: {tag_text}",
        "clip_embedding_path": rec["clip_embedding_path"],
        "source_dataset": rec["source_dataset"],
        "metadata": rec.get("metadata", {})
    })

print("Built retrieval units:", len(retrieval_units))
print_sample("Retrieval unit sample", retrieval_units[0])

Built retrieval units: 7182
Retrieval unit sample
--------------------------------------------------------------------------------
{
  "unit_id": "panorama_9affe9444ceb_chunk_0",
  "record_id": "panorama_9affe9444ceb",
  "modality": "text",
  "retrieval_text": "Name: Declan  \nLocation: Lake Stuartview  \nTimestamp: 2023-09-20 11:13  \nReading about innovative design approaches always sparks ideas in my own projects. Growing up in Lake Stuartview and now working in Boothfort gives me a unique perspective on integrating regional influences with modern trends. Excellent post!",
  "source_dataset": "PANORAMA",
  "metadata": {}
}



In [34]:
save_jsonl(retrieval_units, OUT_DIR / "all_retrieval_units.jsonl")
print("Saved all_retrieval_units.jsonl")

Saved all_retrieval_units.jsonl


## Phase 5: Chroma Ingestion
- Text retrieval units → `text-embedding-3-small` → Chroma `text_collection`
- Image retrieval units → CLIP embeddings (already computed) → Chroma `image_collection`

In [35]:
import os
from openai import OpenAI
import chromadb
from google.colab import userdata
OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")

openai_client = OpenAI(api_key=OPENAI_API_KEY)

TEXT_EMBED_MODEL = "text-embedding-3-small"
TEXT_EMBED_DIM   = 1536   # fixed output dim for text-embedding-3-small
CLIP_EMBED_DIM   = 512    # openai/clip-vit-base-patch32

CHROMA_DIR = OUT_DIR / "chroma_db"
CHROMA_DIR.mkdir(exist_ok=True)

chroma_client = chromadb.PersistentClient(path=str(CHROMA_DIR))
print("Chroma persistent client initialised at:", CHROMA_DIR)

Chroma persistent client initialised at: /content/drive/MyDrive/cs466_baseline_demo/outputs/chroma_db


In [36]:
# get_or_create so the cell is safe to re-run
text_collection = chroma_client.get_or_create_collection(
    name="text_collection",
    metadata={"hnsw:space": "cosine"},
)

image_collection = chroma_client.get_or_create_collection(
    name="image_collection",
    metadata={"hnsw:space": "cosine"},
)

print("text_collection  items before ingestion:", text_collection.count())
print("image_collection items before ingestion:", image_collection.count())

text_collection  items before ingestion: 0
image_collection items before ingestion: 0


In [37]:
def embed_texts_openai(texts: list[str], batch_size: int = 100) -> list[list[float]]:
    """Call text-embedding-3-small in batches; returns list of float vectors."""
    all_embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i : i + batch_size]
        response = openai_client.embeddings.create(
            model=TEXT_EMBED_MODEL,
            input=batch,
        )
        all_embeddings.extend([item.embedding for item in response.data])
    return all_embeddings

In [39]:
text_units = [u for u in retrieval_units if u["modality"] == "text"]
print(f"Text retrieval units to ingest: {len(text_units)}")

INGEST_BATCH = 100

for i in tqdm(range(0, len(text_units), INGEST_BATCH), desc="Ingesting text"):
    batch = text_units[i : i + INGEST_BATCH]

    texts = [u["retrieval_text"] for u in batch]
    embeddings = embed_texts_openai(texts)

    text_collection.upsert(
        ids=[u["unit_id"] for u in batch],
        embeddings=embeddings,
        documents=texts,
        metadatas=[
            {
                "record_id":     u["record_id"],
                "source_dataset": u["source_dataset"],
                **{k: str(v) for k, v in u.get("metadata", {}).items()},
            }
            for u in batch
        ],
    )

print("text_collection items after ingestion:", text_collection.count())

Text retrieval units to ingest: 4111


Ingesting text:   0%|          | 0/42 [00:00<?, ?it/s]

text_collection items after ingestion: 4111


In [40]:
image_units = [u for u in retrieval_units if u["modality"] == "image"]
print(f"Image retrieval units to ingest: {len(image_units)}")

for u in tqdm(image_units, desc="Ingesting images"):
    emb_path = u.get("clip_embedding_path")
    if not emb_path or not Path(emb_path).exists():
        print(f"  WARNING: embedding missing for {u['unit_id']}, skipping")
        continue

    emb = np.load(emb_path).tolist()   # shape (512,) → list[float]

    image_collection.upsert(
        ids=[u["unit_id"]],
        embeddings=[emb],
        documents=[u["retrieval_text"]],
        metadatas=[{
            "record_id":      u["record_id"],
            "source_dataset": u["source_dataset"],
            **{k: str(v) for k, v in u.get("metadata", {}).items()},
        }],
    )

print("image_collection items after ingestion:", image_collection.count())

Image retrieval units to ingest: 3071


Ingesting images:   0%|          | 0/3071 [00:00<?, ?it/s]

image_collection items after ingestion: 3056


In [42]:
## for tensor objects weird bug
@torch.no_grad()
def encode_text_with_clip(text: str):
    inputs = clip_processor(text=[text], return_tensors="pt", padding=True)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    text_features = clip_model.get_text_features(**inputs)

    if not isinstance(text_features, torch.Tensor):
        if hasattr(text_features, "text_embeds"):
            text_features = text_features.text_embeds
        elif hasattr(text_features, "pooler_output"):
            text_features = text_features.pooler_output
        elif hasattr(text_features, "last_hidden_state"):
            text_features = text_features.last_hidden_state[:, 0, :]
        else:
            raise TypeError(f"Unexpected output type: {type(text_features)}")

    text_features = text_features / text_features.norm(dim=-1, keepdim=True)
    return text_features[0].cpu().numpy()

In [44]:
# ---------- quick smoke tests ----------
test_query = "personal data shared without consent"

# text collection query
q_emb = embed_texts_openai([test_query])[0]
text_results = text_collection.query(query_embeddings=[q_emb], n_results=3)
print("=== Top-3 text results ===")
for doc, meta in zip(text_results["documents"][0], text_results["metadatas"][0]):
    print(f"  [{meta['source_dataset']}] {doc[:120]}...")

# image collection query (using CLIP text embedding for cross-modal retrieval)
with torch.no_grad():
    tok = clip_processor(text=[test_query], return_tensors="pt", padding=True)
    tok = {k: v.to(device) for k, v in tok.items()}

    txt_feat = clip_model.get_text_features(**tok)

    # make sure txt_feat is a tensor
    if not isinstance(txt_feat, torch.Tensor):
        # fallback if model returns an output object
        if hasattr(txt_feat, "text_embeds"):
            txt_feat = txt_feat.text_embeds
        elif hasattr(txt_feat, "pooler_output"):
            txt_feat = txt_feat.pooler_output
        elif hasattr(txt_feat, "last_hidden_state"):
            txt_feat = txt_feat.last_hidden_state[:, 0, :]
        else:
            raise TypeError(f"Unexpected CLIP text output type: {type(txt_feat)}")

    txt_feat = txt_feat / txt_feat.norm(dim=-1, keepdim=True)
    clip_query_emb = txt_feat[0].cpu().numpy().tolist()

image_results = image_collection.query(query_embeddings=[clip_query_emb], n_results=3)
print("\n=== Top-3 image results ===")
for doc, meta in zip(image_results["documents"][0], image_results["metadatas"][0]):
    print(f"  [{meta['source_dataset']}] {doc}")

=== Top-3 text results ===
  [PANORAMA] @cheryl_74 : Sharing a recent log from my kitchen network: "Emergency Contact Michael Ferrell notified at +63 585409212 ...
  [PANORAMA] @dasgupta_73 : I’ve been working at Sundaram Inc for decades and now questioning if my company email protocol is secure ...
  [PANORAMA] I'm logging a peculiar error seen during a routine checkup: a merging of personal info fields (like my email garry.watso...

=== Top-3 image results ===
  [VISPR] VISPR image with tags: a16_race, a17_color, a1_age_approx, a33_student_id, a4_gender, a5_eye_color, a6_hair_color, a82_date_time, a9_face_complete
  [VISPR] VISPR image with tags: a0_safe
  [BIV-Priv] BIV-Priv image with tags: image


In [45]:
summary = {
    "num_panorama_text_records": len(text_records),
    "num_datasir_text_records": len(datasir_records),
    "num_total_text_records": len(all_text_records),
    "num_image_records": len(image_records),
    "num_retrieval_units": len(retrieval_units),
    "embedding_dir": str(EMB_DIR),
    "output_dir": str(OUT_DIR),
}

print_sample("Baseline preprocessing summary", summary)

Baseline preprocessing summary
--------------------------------------------------------------------------------
{
  "num_panorama_text_records": 2000,
  "num_datasir_text_records": 1981,
  "num_total_text_records": 3981,
  "num_image_records": 3071,
  "num_retrieval_units": 7182,
  "embedding_dir": "/content/drive/MyDrive/cs466_baseline_demo/outputs/image_embeddings",
  "output_dir": "/content/drive/MyDrive/cs466_baseline_demo/outputs"
}

